# Figure PS02 — Phage case study: the genome with the most candidates (Agrobacterium phage Atu_ph07, NC_042013)

**Status:** draft (Setonix integrator). Answers George's review comment asking for a gene-map case study of the genome with the most candidates.

**Caption (draft).** Agrobacterium phage Atu_ph07 (RefSeq NC_042013; 490,380 bp; 826 Pharokka + phold CDS) holds the most candidates of the 36,025 Inphared genomes: 28 unannotated ORFs ≥ 90 aa with out-of-fold coding probability p ≥ 0.8 in the searched contexts (17 antisense, 10 other-frame, 1 intergenic). Its GenBank original (MF403008) is in the same 95% ANI cluster and ranks second with 26.
**(a)** The whole genome. Annotated CDS on each strand, coloured by PHROG function category; candidates (above the axis for +, below for −) shaped by context (circle, antisense; square, other frame; diamond, intergenic) and filled when Foldseek (3Di + amino acids) finds a full-length hit (query and target coverage ≥ 0.8, E ≤ 10⁻³) in PDB100, AFDB/Swiss-Prot, CATH50, BFVD or phold.
**(b)** The 8 kb around orf104 and orf108. Top, annotated CDS; below, every ORF ≥ 90 aa in its reading frame (as numbered by the ORF caller, as in Fig. 6d), coloured by p; candidates outlined. Both are antisense only by a few codons: each overlaps an opposite-strand gene by 3–6% of its length. orf108 (102 aa) has full-length hits to PAAR-repeat proteins in PDB100 and CATH50 (4JIW, 35% identity), phold and BFVD; orf104 matches a protein of this phage in the submitters' annotation (BFVD, 96% identity).


## Provenance

| item | value |
|---|---|
| plotting data | `../data/figPS02_atu_ph07_{cds,orfs,hits}.tsv.gz`; sources and sha256 in `../data/phage_case_provenance.json` |
| produced by | `manuscript/prep/phage_case_study_prepare_data.py` (copies values; nothing re-estimated) |
| upstream | Setonix `work/phage_rank_all/case_study/` from `Validation/Inphared/scripts/11_case_study.{py,slurm}` (job 51710801): the all-genome ranking (36,025 genomes, genome-grouped CV, intrinsic features) and Foldseek 10.941cd33 against PDB100, AFDB/Swiss-Prot, CATH50, BFVD and phold DB v1.0.0 |
| choice of genome | most candidates in the searched contexts (`Validation/Inphared/all_genomes/tables/candidates_per_genome.tsv`). OR354875 has more p ≥ 0.8 ORFs (30) but 27 are its own annotated genes left unmatched because the sequence holds 2,332 ambiguity codes |

## Data dictionary

| file | columns used | meaning |
|---|---|---|
| `figPS02_atu_ph07_cds` | `g_start`, `g_end` (0-based half-open), `strand`, `function`, `product` | Pharokka + phold CDS |
| `figPS02_atu_ph07_orfs` | `orf_id`, `g_start`, `g_end`, `strand`, `p`, `in_genbank`, `candidate`, `context`, `cds_overlap_frac` | every stop-to-stop ORF ≥ 90 aa; `candidate` = unannotated, p ≥ 0.8, intergenic/antisense/frameshift. Frame lanes follow `get_orfs`: + frame = g_start mod 3 + 1, − frame = (genome length − g_end) mod 3 + 1, the `frame` column that Fig. 6d plots |
| `figPS02_atu_ph07_hits` | `orf_id`, `db`, `fident`, `taxname`, `theader` | best full-length structural hit per candidate and database |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors, cm
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrow, Patch

import msfig

msfig.style()
SLUG = "figPS02_atu_ph07"
C = msfig.PALETTE

In [ ]:
cds = msfig.data("figPS02_atu_ph07_cds.tsv.gz")
orfs = msfig.data("figPS02_atu_ph07_orfs.tsv.gz")
hits = msfig.data("figPS02_atu_ph07_hits.tsv.gz")
cand = orfs[orfs.candidate]

# The numbers the caption and text quote.
L = int(cds.record_length.iloc[0])
assert L == 490380 and len(cds) == 826
assert len(cand) == 28
assert cand.context.value_counts().to_dict() == {"antisense": 17, "frameshift": 10, "intergenic": 1}
assert hits.orf_id.nunique() == 5
h108 = hits[hits.orf_id == "orf108"].set_index("db")
assert set(h108.index) == {"pdb100", "cath50", "phold", "bfvd"}
assert round(h108.loc["pdb100", "fident"], 2) == 0.35 and "PAAR" in h108.loc["pdb100", "theader"]
assert round(hits.set_index(["orf_id", "db"]).loc[("orf104", "bfvd"), "fident"], 2) == 0.96
assert int(orfs.set_index("orf_id").loc["orf108", "protein_length"]) == 102
# orf104 and orf108 are antisense only by a few codons
assert orfs.set_index("orf_id").loc[["orf104", "orf108"], "cds_overlap_frac"].between(0.03, 0.06).all()

In [ ]:
FUNC = {"DNA, RNA and nucleotide metabolism": C["blue"], "head and packaging": C["orange"],
        "tail": C["green"], "connector": C["purple"], "lysis": C["vermillion"],
        "moron, auxiliary metabolic gene and host takeover": C["yellow"],
        "transcription regulation": C["sky"], "integration and excision": C["black"],
        "other": "#7f7f7f", "unknown function": "#d9d9d9"}
# candidates are black and coded by shape: every palette colour is a PHROG category above
CTX = {"antisense": "o", "frameshift": "s", "intergenic": "D"}
with_hit = set(hits.orf_id)

fig = plt.figure(figsize=(7.2, 5.6))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.55], hspace=0.95)

# (a) whole genome
ax = fig.add_subplot(gs[0])
for c in cds.itertuples():
    y = 0.3 if c.strand == "+" else -0.3
    ax.add_patch(plt.Rectangle((c.g_start, y - 0.18), c.g_end - c.g_start, 0.36, lw=0,
                               fc=FUNC.get(c.function, "#d9d9d9")))
for o in cand.itertuples():
    mid = (o.g_start + o.g_end) / 2
    up = o.strand == "+"
    filled = o.orf_id in with_hit
    ax.plot(mid, 0.85 if up else -0.85, marker=CTX[o.context], ms=5, mec="black",
            mfc="black" if filled else "white", mew=0.9, ls="none")
o = cand.set_index("orf_id").loc["orf108"]
ax.annotate("orf108 (PAAR)", ((o.g_start + o.g_end) / 2, 0.85 if o.strand == "+" else -0.85),
            xytext=(0, 9), textcoords="offset points", ha="center", fontsize=6.5)
ax.axhline(0, color="black", lw=0.5)
ax.set_xlim(0, L)
ax.set_ylim(-1.3, 1.35)
ax.set_yticks([0.85, 0.3, -0.3, -0.85], ["candidates +", "CDS +", "CDS −", "candidates −"])
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x / 1000:.0f}"))
ax.set_xlabel("position in NC_042013 (kb)")
ax.set_title("a  Agrobacterium phage Atu_ph07: 826 annotated genes and 28 candidates", loc="left",
             fontweight="bold")
used = [f for f in FUNC if f in set(cds.function)]
h1 = [Patch(fc=FUNC[f], label=f.replace("moron, auxiliary metabolic gene and host takeover", "moron / AMG"))
      for f in used]
h2 = [Line2D([], [], marker=v, ls="none", mec="black", mfc="black", label=k.replace("frameshift", "other frame"))
      for k, v in CTX.items()]
h2.append(Line2D([], [], marker="o", ls="none", mec="black", mfc="white", label="no full-length hit"))
ax.legend(handles=h1 + h2, ncol=5, fontsize=5.8, loc="upper center", bbox_to_anchor=(0.5, -0.42),
          handlelength=1.2, columnspacing=0.9)

# (b) zoom
ax = fig.add_subplot(gs[1])
x0, x1 = 341500, 349500
cmap = plt.get_cmap("cividis")
norm = colors.Normalize(0, 1)


def arrow(s, e, strand, yc, fc, ec, lw=0.6, h=0.62):
    length = e - s
    head = min(120, length * 0.35)
    start, dx = (s, length) if strand == "+" else (e, -length)
    ax.add_patch(FancyArrow(start, yc, dx, 0, width=h, head_width=h, head_length=head,
                            length_includes_head=True, fc=fc, ec=ec, lw=lw))


def lane(o):
    # get_orfs frame numbering, as in Fig. 6d: the minus strand counts from the genome's end
    ph = (o.g_start % 3) if o.strand == "+" else ((L - o.g_end) % 3)
    return {"+": 5 - ph, "-": 1 - ph}[o.strand]


win = cds[(cds.g_end > x0) & (cds.g_start < x1)]
for c in win.itertuples():
    arrow(max(c.g_start, x0), min(c.g_end, x1), c.strand, 6.6, FUNC.get(c.function, "#d9d9d9"), "#7f7f7f")
zw = orfs[(orfs.g_end > x0) & (orfs.g_start < x1)]
for o in zw.itertuples():
    if o.candidate:
        continue
    arrow(max(o.g_start, x0), min(o.g_end, x1), o.strand, lane(o), cmap(norm(o.p)),
          "black" if o.in_genbank else "none", lw=0.8)
for o in zw[zw.candidate].itertuples():
    yc = lane(o)
    arrow(o.g_start, o.g_end, o.strand, yc, cmap(norm(o.p)), C["vermillion"], lw=1.5)
    lab = {"orf108": "orf108  PAAR-repeat protein\nPDB 4JIW, CATH, phold, BFVD",
           "orf104": "orf104  Atu_ph07 protein (BFVD 96%)"}.get(o.orf_id, o.orf_id)
    below = o.orf_id == "orf108"   # the CDS row sits just above lane +1 here
    ax.text((o.g_start + o.g_end) / 2, yc - 0.45 if below else yc + 0.45, lab, ha="center",
            va="top" if below else "bottom", fontsize=6, color=C["vermillion"])
ax.set_xlim(x0, x1)
ax.set_ylim(-1.8, 7.4)
ax.set_yticks([6.6, 5, 4, 3, 1, 0, -1], ["annotated CDS", "+1", "+2", "+3", "−1", "−2", "−3"])
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x / 1000:.1f}"))
ax.set_xlabel("position in NC_042013 (kb)")
ax.set_title("b  orf104 and orf108 in context (black outline: ORF of an annotated gene)",
             loc="left", fontweight="bold")
cb = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.03, pad=0.01)
cb.set_label("coding probability p", fontsize=6.5)
cb.ax.tick_params(labelsize=6)

msfig.save(fig, SLUG)

## Notes

- Candidates exclude unannotated ORFs that share a stop or frame with an annotated gene (pieces of that gene).
- The BFVD hits of orf104, orf844 and orf892 are UniProt translations of this phage's own submitted annotation (83–100% identity): the submitters called these ORFs, and the Pharokka + phold re-annotation did not. orf875 hits a phold singleton PHROG at 100% identity. orf108 is the only candidate with full-length hits outside the viral databases.
- 23 of the 28 candidates have no full-length structural hit in any database.